# AAPL 数据探索与验证结果

本 Notebook 调用项目 `.py` 模块，展示现有数据和已完成的验证集实验，不自动训练模型。

- 训练目标日：2010—2021 年；验证目标日：2022—2023 年。
- 2024—2025 测试期与 2026 保留期不参与本 Notebook。
- 请在编辑器选择项目环境的 Python 内核（默认 **EE6008**），然后从上至下运行。
- 收益为小数单位，0.01 表示 1%；direction 的 0 包含下跌和平盘。


In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Locate the project from either its root or notebooks directory.
# 从项目根目录或 notebooks 子目录定位，不写死机器路径。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/aapl.json").exists()), None)
if ROOT is None:
    raise RuntimeError("请从 stock_prediction 或 notebooks 目录打开此 Notebook")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data import load_dataset
from src.evaluate import comparison_table, plot_comparison

print("Python:", sys.executable)
config = json.loads((ROOT / "configs/aapl.json").read_text(encoding="utf-8-sig"))


## 1. 检查统一数据集

这里复用 `src/data.py` 的特征、标签和切分逻辑。标准化只拟合训练窗口涉及的唯一日期。表格记录实际可用样本和目标日期。


In [ ]:
dataset = load_dataset(config, ROOT)
display(pd.DataFrame([dataset.audit]).T.rename(columns={0: "value"}))
display(dataset.train.dates.head())
display(dataset.validation.dates.head())


## 2. 查看历史输入

仅展示训练和验证期。8 个相对特征使用相同历史窗口，分别以表格和序列形式提供给两个模型。数据源与复权口径仍待组员确认。


In [ ]:
display(dataset.feature_frame.head(6))
fig, ax = plt.subplots(figsize=(12, 4))
history = dataset.feature_frame.set_index("date")
ax.plot(history.index, history["return"] * 100, linewidth=0.6, label="AAPL")
ax.plot(history.index, history["nasdaq_return"] * 100, linewidth=0.6, alpha=0.6, label="Nasdaq")
ax.set(title="Historical daily returns through validation period", ylabel="Daily return (%)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


## 3. 加载已完成的实验

首次使用请先在终端运行 `python main.py`。下方默认选择最近完成的验证实验，也可以把 RUN_NAME 改为指定目录名。分析时以实验保存的 config 为准，修改当前配置不会改变既有实验结果。


In [ ]:
RUN_NAME = None  # 如需固定结果，填写 "validation_initial"
if RUN_NAME:
    run_dir = ROOT / "artifacts" / RUN_NAME
else:
    completed = []
    for path in (ROOT / "artifacts").glob("*/manifest.json"):
        manifest = json.loads(path.read_text(encoding="utf-8"))
        if manifest.get("status") == "complete" and manifest.get("evaluation_split") == "validation":
            completed.append(path)
    if not completed:
        raise RuntimeError("尚无完成的验证实验，请先运行 main.py")
    run_dir = max(completed, key=lambda p: p.stat().st_mtime).parent
manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
if manifest["status"] != "complete" or manifest["evaluation_split"] != "validation":
    raise ValueError("请选择已完成的验证集实验")
metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
run_config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
print("Run:", run_dir.name)
print("Config matches current exploration:", run_config == config)
display(comparison_table(metrics).astype(float).round(6))
reference = json.loads((run_dir / "direction_reference.json").read_text(encoding="utf-8"))
print("训练期多数方向基线：", reference)
fig = plot_comparison(metrics)
plt.show()


## 4. 检查预测与误差

MAE/RMSE 衡量收益数值的误差；方向准确率仅比较符号，两者可能给出不同的模型排序。`r2_vs_zero` 大于零表示平方误差低于零收益预测。

下面查看已保存结果，而不是重新计算一份模型输出。


In [ ]:
predictions = pd.read_csv(run_dir / "predictions.csv", parse_dates=["window_start", "signal_date", "target_date"])
display(predictions.head())
learned = [name for name in ("xgboost", "patchtst") if name in predictions]
if learned:
    fig, axes = plt.subplots(len(learned), 1, figsize=(12, 3 * len(learned)), squeeze=False, layout="constrained")
    recent = predictions.tail(80)
    for ax, name in zip(axes[:, 0], learned):
        ax.plot(recent.target_date, recent.actual_return * 100, label="Actual", linewidth=1)
        ax.plot(recent.target_date, recent[name] * 100, label=name, linewidth=1)
        ax.set(ylabel="Daily return (%)", title=f"{name}: last 80 validation targets")
        ax.legend()
        ax.grid(alpha=0.2)
    plt.show()


## 5. 如何解释结果

- 先比较模型与零收益、训练均值和昨日收益基线，再比较两个主模型。
- 同时查看收益误差、方向准确率、预测上涨比例和混淆矩阵，避免只挑有利指标。
- 本轮只有一个随机种子和固定时间切分，尚未完成稳健性、最终测试或交易成本回测。
- PatchTST 是通道共享编码、联合单收益头的适配版本，使用 LayerNorm，未启用 RevIN 或残差注意力；详细说明见 README。
- 后续实验参数应由训练和验证结果决定，2024 年以后的数据继续封存。
